This code provides a complete implementation of transfer learning with either ResNet50 or EfficientNetB0. Key features include:<br>

Two-phase training (frozen and fine-tuning phases)<br>
Data augmentation for training<br>
Early stopping and model checkpointing<br>
Dropout layers to prevent overfitting<br>
Learning rate reduction for fine-tuning<br>
Visualization of training results<br>

In [ ]:
# Organize your dataset in the following structure

In [ ]:
train/
    class1/
        image1.jpg
        image2.jpg
    class2/
        image3.jpg
        image4.jpg
val/
    class1/
        image5.jpg
    class2/
        image6.jpg


Update the train_dir and val_dir paths in the main function<br>
Choose between ResNet50 or EfficientNetB0 by uncommenting the appropriate line<br>
Run the code<br>
The model will automatically save the best weights during training and the final model after completion.<br>

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50, EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Configuration parameters
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 0.0001


In [ ]:
# Data prep

In [ ]:
def create_data_generators(train_dir, val_dir):
    # Data augmentation for training
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest'
    )
    
    # Only rescaling for validation
    val_datagen = ImageDataGenerator(rescale=1./255)
    
    train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode='categorical'
    )
    
    validation_generator = val_datagen.flow_from_directory(
        val_dir,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode='categorical'
    )
    
    return train_generator, validation_generator


In [ ]:
# Creating transfer learning models (both ResNet50 and EfficientNetB0)

In [ ]:
def create_resnet50_model(num_classes):
    # Load ResNet50 with pre-trained weights
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    
    # Freeze the pre-trained layers
    for layer in base_model.layers:
        layer.trainable = False
    
    # Add custom layers on top
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(1024, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.3)(x)
    predictions = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=base_model.input, outputs=predictions)
    return model, base_model

def create_efficientnet_model(num_classes):
    # Load EfficientNetB0 with pre-trained weights
    base_model = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    
    # Freeze the pre-trained layers
    for layer in base_model.layers:
        layer.trainable = False
    
    # Add custom layers on top
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(1024, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.3)(x)
    predictions = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=base_model.input, outputs=predictions)
    return model, base_model


In [ ]:
# Training function with fine-tuning:

In [ ]:
def train_model(model, base_model, train_generator, validation_generator):
    # First phase: training only the top layers
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    # Callbacks
    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    )
    
    checkpoint = ModelCheckpoint(
        'best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max'
    )
    
    # First phase of training
    print("Training top layers...")
    history1 = model.fit(
        train_generator,
        steps_per_epoch=train_generator.samples // BATCH_SIZE,
        validation_data=validation_generator,
        validation_steps=validation_generator.samples // BATCH_SIZE,
        epochs=10,
        callbacks=[early_stopping, checkpoint]
    )
    
    # Second phase: fine-tuning
    # Unfreeze some layers of the base model
    if isinstance(base_model, ResNet50):
        for layer in base_model.layers[-20:]:
            layer.trainable = True
    else:  # EfficientNetB0
        for layer in base_model.layers[-30:]:
            layer.trainable = True
    
    # Recompile the model with a lower learning rate
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE/10),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    # Second phase of training
    print("Fine-tuning...")
    history2 = model.fit(
        train_generator,
        steps_per_epoch=train_generator.samples // BATCH_SIZE,
        validation_data=validation_generator,
        validation_steps=validation_generator.samples // BATCH_SIZE,
        epochs=EPOCHS,
        callbacks=[early_stopping, checkpoint]
    )
    
    return history1, history2


In [ ]:
#Main execution code

In [ ]:
def main():
    # Set paths to your data
    train_dir = 'path/to/train/data'
    val_dir = 'path/to/validation/data'
    
    # Create data generators
    train_generator, validation_generator = create_data_generators(train_dir, val_dir)
    
    # Get number of classes
    num_classes = len(train_generator.class_indices)
    
    # Choose model type (uncomment the one you want to use)
    model, base_model = create_resnet50_model(num_classes)
    # model, base_model = create_efficientnet_model(num_classes)
    
    # Train the model
    history1, history2 = train_model(
        model,
        base_model,
        train_generator,
        validation_generator
    )
    
    # Save the final model
    model.save('final_model.h5')
    
    return model, history1, history2

# Function to make predictions
def predict_image(model, image_path):
    img = tf.keras.preprocessing.image.load_img(
        image_path,
        target_size=(IMG_SIZE, IMG_SIZE)
    )
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)
    img_array = img_array / 255.0
    
    predictions = model.predict(img_array)
    return predictions

if __name__ == "__main__":
    model, history1, history2 = main()


In [ ]:
# Visualizing the training results

In [ ]:
import matplotlib.pyplot as plt

def plot_training_history(history1, history2):
    # Combine histories
    acc = history1.history['accuracy'] + history2.history['accuracy']
    val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
    loss = history1.history['loss'] + history2.history['loss']
    val_loss = history1.history['val_loss'] + history2.history['val_loss']
    
    epochs_range = range(len(acc))
    
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.legend()
    plt.title('Training and Validation Accuracy')
    
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.legend()
    plt.title('Training and Validation Loss')
    
    plt.tight_layout()
    plt.show()
